# Importing an `snnTorch` SNN model to LAVA

In [6]:
# Show current directory
import os
curr_dir = os.getcwd()
print(curr_dir)

# Check if the current WD is the file location
if "/lava-dl/src/nir" not in os.getcwd():
    # Set working directory to this file location
    file_location = f"{os.getcwd()}/lava-dl/src/nir"
    print("File Location: ", file_location)

    # Change the current working Directory
    os.chdir(file_location)

    # New Working Directory
    print("New Working Directory: ", os.getcwd())

/home/monkin/Desktop/feup/thesis/lava-dl/src/nir


In [7]:
import numpy as np
import nir
import matplotlib.pyplot as plt

nir_network = nir.read("nir_model_cuba.nir")

In [8]:
# Print the network summary
print(f"NIR Network Info: Nº Nodes: {len(nir_network.nodes)} | Nº Edges: {len(nir_network.edges)}")

NIR Network Info: Nº Nodes: 6 | Nº Edges: 5


In [28]:

MANUAL_CREATION = False

if MANUAL_CREATION:
    # TODO: This is not the original code. I changed it because it had errors..
    ng = nir.NIRGraph(
        nodes={
            "input": nir.Input(input_type=np.array([3])),
            # "affine": nir.Affine(weight=np.array([[8, 2, 10], [14, 3, 14]]).T, bias=np.array([1, 2])),
            "lif": nir.CubaLIF(
                # tau=np.array([1] * 2),
                r=np.array([1 / 1e-4] * 2),
                tau_syn=np.array([1] * 2),
                tau_mem=np.array([1] * 2),
                v_leak=np.array([0] * 2),
                v_threshold=np.array([1] * 2),
            ),
            "out": nir.Output(np.array([3])),
        },
        edges=[
            # ("input", "affine"),
            # ("affine", "lif"),
            ("input", "lif"),
            ("lif", "out")
        ],
    )

In [9]:
from nir_to_lava import ImportConfig, LavaLibrary, import_from_nir

config = ImportConfig(
    dt=1e-4, fixed_pt=False, on_chip=False, library_preference=LavaLibrary.LavaDl,
)

In [10]:
nir2lava_network = import_from_nir(nir_network, config)

node 0: Affine
node 1: CubaLIF
node 2: Affine
node 3: CubaLIF
node output: Output


In [11]:
print(nir2lava_network)

NIR2LavaDLNetwork(
  (blocks): ModuleList(
    (0): Dense(784, 500, kernel_size=(1, 1, 1), stride=(1, 1, 1))
    (1): Dense(
      (neuron): Neuron()
      (synapse): Dense(7, 7, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
    )
    (2): Dense(500, 10, kernel_size=(1, 1, 1), stride=(1, 1, 1))
    (3): Dense(
      (neuron): Neuron()
      (synapse): Dense(7, 7, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
    )
  )
)


Under a certain set of parameters, I was able to define a network in `snnTorch`, export it to the `NIR` format and import it to `Lava`. This network used CUBA LIF neurons, `snn.Synaptic` (in snn-torch) or `LIF` (in lava).